In [11]:
import os
import glob

import json
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List, Tuple, Any, Optional
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from concurrent.futures import ProcessPoolExecutor, as_completed

In [12]:
# STORAGE_DIR = "Z:\\starcraft-vision\\"
STORAGE_DIR = "/workspace"

DATA_DIR = os.path.join(STORAGE_DIR, "data")
GROUND_TRUTH_DIR = os.path.join(DATA_DIR, "label", "dst")
PREDICTIONS_DIR = os.path.join(STORAGE_DIR, "predictions")

In [13]:
# REPLAYS = [36, 212, 438, 522, 1660, 6254]
# REPLAYS = [1559, 1628, 2351, 6219, 11251]
# REPLAYS = [275, 1725, 3613, 4520, 4664]
REPLAYS = [36, 212, 438, 522, 1660, 1559, 1628, 2351, 6219, 11251, 275, 1725, 3613, 4520, 4664]

REPLAYS = [str(_) + '.rep' for _ in REPLAYS]

In [14]:
# TARGET_MODEL = "test_input_model/model_020"
# TARGET_MODEL = "all_correct_win4_b16_20250826_045543/model_030"
TARGET_MODEL_1 = "all_correct_win4_b16_20250826_045543/model_030"
TARGET_MODEL_2 = "all_correct_win4_b16_kbrs_20250912_005403/model_030"


In [15]:
ground_truth_json_path = [
    f for name in REPLAYS
    for f in glob.glob(os.path.join(GROUND_TRUTH_DIR, name, '*.json'))
]

print(f"Found {len(ground_truth_json_path)} ground truth files.")

Found 15 ground truth files.


In [26]:
import os, json, mmap
from pathlib import Path
from typing import Any, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ---------- 경로 수집 (빠른 scandir 재귀) ----------
def _iter_json_paths(root: str) -> List[str]:
    out = []
    stack = [os.path.abspath(root)]
    while stack:
        d = stack.pop()
        try:
            with os.scandir(d) as it:
                for e in it:
                    if e.is_dir(follow_symlinks=False):
                        stack.append(e.path)
                    elif e.is_file() and e.name.endswith(".json"):
                        out.append(e.path)
        except PermissionError:
            pass
    return out

def _build_json_paths(directory: str,
                      label_method: str = "all_correct",
                      replays: Optional[List[str]] = None) -> List[str]:
    directory = os.path.abspath(directory)
    if replays:
        paths = [os.path.join(directory, rep, f"{label_method}.json") for rep in replays]
        # 존재하는 파일만
        return [p for p in paths if os.path.isfile(p)]
    # 전체 스캔
    return _iter_json_paths(directory)

# ---------- 빠른 로더 (orjson + mmap, fallback 표준 json) ----------
import os, json, mmap, gzip

def _read_one_json(path: str, use_orjson: bool = True):
    try:
        size = os.path.getsize(path)
        if size == 0:
            return path, None, "EmptyFileError: file size is 0"

        if use_orjson:
            try:
                import orjson
                with open(path, "rb") as f:
                    # gzip 여부 감지 (0x1f 0x8b)
                    head = f.read(2)
                    f.seek(0)
                    if head == b"\x1f\x8b":
                        with gzip.GzipFile(fileobj=f) as gz:
                            b = gz.read()  # bytes
                            return path, orjson.loads(b), None
                    # mmap + memoryview 로 zero-copy 파싱
                    with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
                        return path, orjson.loads(memoryview(mm)), None
            except ModuleNotFoundError:
                pass  # orjson 없으면 표준 json으로 폴백

        # 표준 json 폴백 (BOM 처리 위해 utf-8-sig)
        with open(path, "r", encoding="utf-8-sig") as f:
            return path, json.load(f), None

    except Exception as e:
        return path, None, f"{type(e).__name__}: {e}"


# ---------- 자동 전략 선택 ----------
def _choose_strategy(paths: List[str], bytes_threshold: int = 512_000) -> str:
    """
    파일 크기 중앙값이 threshold 이상이면 'process', 아니면 'thread'
    - 작은 파일: 스레드가 유리(I/O 병목 + orjson이 GIL 거의 점유 안함)
    - 큰 파일: 프로세스가 유리(파싱 CPU 부담 분산)
    """
    if not paths:
        return "thread"
    sizes = []
    step = max(1, len(paths)//200)  # 샘플링
    for p in paths[::step]:
        try:
            sizes.append(os.path.getsize(p))
        except OSError:
            pass
    if not sizes:
        return "thread"
    from statistics import median
    return "process" if median(sizes) >= bytes_threshold else "thread"

# ---------- 공통 실행기 ----------
def read_json_files_fast(directory: str,
                         label_method: str = "all_correct",
                         replays: Optional[List[str]] = None,
                         max_workers: Optional[int] = None,
                         force: Optional[str] = None  # 'thread'|'process'|None(auto)
                         ) -> List[Any]:
    paths = _build_json_paths(directory, label_method, replays)
    if not paths:
        return []

    # 전략/워커 수 결정
    strategy = force or _choose_strategy(paths)
    if max_workers is None:
        cpu = os.cpu_count() or 4
        if strategy == "process":
            max_workers = min(cpu, 8)            # 과도한 프로세스는 오버헤드↑
        else:
            max_workers = min(32, cpu * 2)       # 작은 파일 다수에 적절

    results: List[Any] = []
    errors: List[str] = []

    if strategy == "process":
        # ProcessPool은 chunksize가 중요 (작은 태스크 묶음)
        chunksize = max(1, len(paths) // (max_workers * 8))
        with ProcessPoolExecutor(max_workers=max_workers) as ex:
            for path, data, err in ex.map(_read_one_json, paths, chunksize=chunksize):
                if err:
                    errors.append(f"[FAIL] {path} -> {err}")
                else:
                    results.append(data)
    else:
        # ThreadPool: orjson이 C에서 돌며 GIL을 잘 놓기 때문에 효율적
        with ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix="json") as ex:
            for path, data, err in ex.map(_read_one_json, paths):
                if err:
                    errors.append(f"[FAIL] {path} -> {err}")
                else:
                    results.append(data)

    if errors:
        print("\n".join(errors))
    return results


In [27]:
ground_truth_data = read_json_files_fast(
    GROUND_TRUTH_DIR,
    label_method='all_correct', replays=REPLAYS,
    max_workers=16,
    force='process'
)

In [28]:
# 작은 JSON 다수면 thread가, 큰 JSON(수 MB+)이면 process가 유리
prediction_vanilla_data = read_json_files_fast(
    os.path.join(PREDICTIONS_DIR, TARGET_MODEL_1),
    label_method='all_correct', replays=REPLAYS,
    max_workers=16,             
    force='process'
)
prediction_kbrs_data = read_json_files_fast(
    os.path.join(PREDICTIONS_DIR, TARGET_MODEL_2),
    label_method='all_correct', replays=REPLAYS,
    max_workers=16,
    force='process'
)

In [29]:
def get_annotations(data, replay_suffix=".rep"):
    frames = []

    for item in (data or []):
        anns = item.get("annotations", [])
        if not anns:
            continue

        df = pd.DataFrame(anns)

        desc = (item.get("info") or {}).get("description", "") or ""
        replay_id = None
        if isinstance(desc, str) and desc.strip():
            replay_id = desc.strip().split()[-1]

        if replay_id is None:
            replay_id = "unknown"

        if replay_suffix and not str(replay_id).endswith(replay_suffix):
            replay_id = f"{replay_id}{replay_suffix}"

        df["replay"] = replay_id
        frames.append(df)

    base_cols = ["replay", "id", "image_id", "category_id", "bbox", "area", "segmentation", "iscrowd"]
    if not frames:
        return pd.DataFrame(columns=base_cols + ["score"])

    has_score = any("score" in f.columns for f in frames)

    wanted_cols = base_cols + (["score"] if has_score else [])
    for i, f in enumerate(frames):
        for col in wanted_cols:
            if col not in f.columns:
                frames[i][col] = pd.NA

    annotations = pd.concat(frames, ignore_index=True)

    return annotations[wanted_cols]

In [30]:
annotation_ground_truth = get_annotations(ground_truth_data)
annotation_prediction_vanilla = get_annotations(prediction_vanilla_data)
annotation_prediction_kbrs = get_annotations(prediction_kbrs_data)

In [31]:
def dataframe_topk_score(dataframe, K=1):
    topk_df = (
      dataframe.sort_values('score', ascending=False)
        .groupby(['replay', 'image_id'], group_keys=False)
        .head(K)
    )
    return topk_df.sort_values(['replay', 'image_id'])

In [32]:
top1_annotation_prediction_vanilla = dataframe_topk_score(annotation_prediction_vanilla, K=1)
top1_annotation_prediction_kbrs = dataframe_topk_score(annotation_prediction_kbrs, K=1)

In [33]:
annotation_ground_truth["idx"] = annotation_ground_truth.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_ground_truth = annotation_ground_truth.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_ground_truth.columns = [f"gt_{i}" for i in wide_ground_truth.columns]
wide_ground_truth.reset_index(inplace=True)


In [34]:
top1_annotation_prediction_vanilla["idx"] = top1_annotation_prediction_vanilla.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction_vanilla = top1_annotation_prediction_vanilla.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction_vanilla.columns = [f"pred_{i}" for i in wide_prediction_vanilla.columns]
wide_prediction_vanilla.reset_index(inplace=True)

top1_annotation_prediction_kbrs["idx"] = top1_annotation_prediction_kbrs.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction_kbrs = top1_annotation_prediction_kbrs.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction_kbrs.columns = [f"pred_{i}" for i in wide_prediction_kbrs.columns]
wide_prediction_kbrs.reset_index(inplace=True)

In [35]:
result_inference_vanilla = pd.merge(wide_ground_truth, wide_prediction_vanilla,on=["replay", "image_id"], how="outer")
result_inference_vanilla = result_inference_vanilla.bfill()
result_inference_vanilla = result_inference_vanilla.dropna()

new_cols = []

for c in result_inference_vanilla.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback

result_inference_vanilla.columns = pd.MultiIndex.from_tuples(new_cols)

result_inference_vanilla.head()

meta                         gt                                      \
      replay image_id                 0                 1                 2   
0  11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                     pred  
                  3                 4                   0  
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]

In [36]:
result_inference_kbrs = pd.merge(wide_ground_truth, wide_prediction_kbrs,on=["replay", "image_id"], how="outer")
result_inference_kbrs = result_inference_kbrs.bfill()
result_inference_kbrs = result_inference_kbrs.dropna()

new_cols = []
for c in result_inference_kbrs.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback
        
result_inference_kbrs.columns = pd.MultiIndex.from_tuples(new_cols)
result_inference_kbrs.head()

meta                         gt                                      \
      replay image_id                 0                 1                 2   
0  11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                 pred  
                  3                 4               0  
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]

In [37]:
def union_metrics_singlepass(rows, gt_cols, pred_cols,
                             grid_w, grid_h, orig_w, orig_h,
                             kernel_x, kernel_y, use_bbox_size, thresholds,
                             return_summary: bool = False):
    n = len(rows)
    ic_multi = np.full(n, np.nan, dtype=float)
    ic_ratio = np.zeros(n, dtype=float)
    ic_flags = {t: np.zeros(n, dtype=float) for t in thresholds}

    def xywh_to_rect(x,y,w,h):
        if not use_bbox_size:
            w,h = kernel_x, kernel_y
        rx = int(round(x * 32 / orig_w * (grid_w - w)))
        ry = int(round(y * 32 / orig_h * (grid_h - h)))
        rx = max(0, min(grid_w - w, rx))
        ry = max(0, min(grid_h - h, ry))
        return rx, ry, w, h

    for j, (_, row) in enumerate(rows.iterrows()):
        total = np.zeros((grid_h, grid_w), dtype=np.int16)
        have_gt = False
        for gc in gt_cols:
            if gc in row and row[gc] is not None and not (isinstance(row[gc], float) and np.isnan(row[gc])):
                x,y,w,h = (row[gc] if len(row[gc])==4 else tuple(row[gc])[:4])
                rx,ry,rw,rh = xywh_to_rect(x,y,w,h)
                total[ry:ry+rh, rx:rx+rw] += 1
                have_gt = True

        pred_mask = np.zeros((grid_h, grid_w), dtype=bool)
        for pc in pred_cols:
            if pc in row and row[pc] is not None and not (isinstance(row[pc], float) and np.isnan(row[pc])):
                x,y,w,h = (row[pc] if len(row[pc])==4 else tuple(row[pc])[:4])
                rx,ry,rw,rh = xywh_to_rect(x,y,w,h)
                pred_mask[ry:ry+rh, rx:rx+rw] = True

        if not have_gt or not pred_mask.any():
            ic_multi[j] = np.nan
            ic_ratio[j] = 0.0
            for t in thresholds:
                ic_flags[t][j] = 0.0
            continue

        vals = total[pred_mask]
        ic_multi[j] = vals.mean()
        ic_ratio[j] = (vals > 0).mean()
        for t in thresholds:
            ok = (ic_ratio[j] > t) if t == 0.0 else (ic_ratio[j] >= t)
            ic_flags[t][j] = float(ok)

    out = pd.DataFrame({
        ("union","ic_multi"): ic_multi,
        ("union","ic_ratio"): ic_ratio,
        **{("union", f"ic@{int(round(t*100)):03d}"): ic_flags[t] for t in thresholds}
    }, index=rows.index)

    return out

In [38]:
def _union_metrics_worker(args):
    (sub_df, gt_cols, pred_cols, grid_w, grid_h, orig_w, orig_h,
     kernel_x, kernel_y, use_bbox_size, thresholds) = args
    return union_metrics_singlepass(sub_df, gt_cols, pred_cols, grid_w, grid_h,
                                    orig_w, orig_h, kernel_x, kernel_y,
                                    use_bbox_size, thresholds)

In [39]:
def add_union_metrics_parallel(
    df,
    gt_cols=(("gt","0"),("gt","1"),("gt","2"),("gt","3"),("gt","4")),
    pred_cols=(("pred","0"),),
    grid_w=128, grid_h=128,
    orig_w=3456, orig_h=3720,
    kernel_x=20, kernel_y=12,
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5),
    max_workers=None, chunksize=None
):
    out_cols = [("union","ic_multi"), ("union","ic_ratio")] + \
               [("union", f"ic@{int(round(t*100)):03d}") for t in thresholds]
    for c in out_cols:
        if c not in df.columns:
            df[c] = np.nan

    idx = df.index.to_numpy()
    if chunksize is None:
        nworkers = max_workers or os.cpu_count() or 4
        chunksize = max(1000, len(idx)//(nworkers*4) or 1)
    splits = [idx[i:i+chunksize] for i in range(0, len(idx), chunksize)]

    arglist = [(df.loc[s].copy(), gt_cols, pred_cols, grid_w, grid_h,
                orig_w, orig_h, kernel_x, kernel_y, use_bbox_size, thresholds)
               for s in splits]

    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_union_metrics_worker, a) for a in arglist]
        for f in as_completed(futs):
            results.append(f.result())

    out = pd.concat(results).sort_index()
    df.loc[out.index, out.columns] = out.values
    return df

In [40]:
out_vanilla = add_union_metrics_parallel(
    result_inference_vanilla,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

out_vanilla

meta                         gt                    \
           replay image_id                 0                 1   
0       11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]   
...           ...      ...               ...               ...   
387678   4664.rep    20690  [108, 2, 20, 12]  [108, 2, 20, 12]   
387679   4664.rep    20691  [108, 2, 20, 12]  [108, 2, 20, 12]   
387680   4664.rep    20692  [108, 2, 20, 12]  [108, 2, 20, 12]   
387681   4664.rep    20693  [108, 2, 20, 12]  [108, 2, 20, 12]   
387682   4664.rep    20694  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                              \
                       2                 3                 4   
0       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
...                  ...               ...               ...   
387678  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387679  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387680  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387681  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387682  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                      pred    union                                
                         0 ic_multi ic_ratio ic@000 ic@030 ic@050  
0       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
1       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
2       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
3       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
4       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
...                    ...      ...      ...    ...    ...    ...  
387678    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387679    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387680    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387681    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387682    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  

[387683 rows x 13 columns]

In [41]:
out_kbrs = add_union_metrics_parallel(
    result_inference_kbrs,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

out_kbrs                                          

meta                          gt                    \
           replay image_id                  0                 1   
0       11251.rep        0   [108, 2, 20, 12]  [108, 2, 20, 12]   
1       11251.rep        1   [108, 2, 20, 12]  [108, 2, 20, 12]   
2       11251.rep        2   [108, 2, 20, 12]  [108, 2, 20, 12]   
3       11251.rep        3   [108, 2, 20, 12]  [108, 2, 20, 12]   
4       11251.rep        4   [108, 2, 20, 12]  [108, 2, 20, 12]   
...           ...      ...                ...               ...   
452355   6219.rep    36788  [104, 48, 20, 12]    [0, 1, 20, 12]   
452356   6219.rep    36789  [104, 48, 20, 12]    [0, 1, 20, 12]   
452357   6219.rep    36790  [104, 48, 20, 12]    [0, 1, 20, 12]   
452358   6219.rep    36791  [104, 48, 20, 12]    [0, 1, 20, 12]   
452359   6219.rep    36792  [104, 48, 20, 12]    [0, 1, 20, 12]   

                                                                 \
                         2                 3                  4   
0         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
1         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
2         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
3         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
4         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
...                    ...               ...                ...   
452355  [100, 115, 20, 12]  [95, 51, 20, 12]  [102, 91, 20, 12]   
452356  [100, 115, 20, 12]  [95, 51, 20, 12]  [101, 90, 20, 12]   
452357  [100, 115, 20, 12]  [95, 51, 20, 12]  [100, 89, 20, 12]   
452358  [100, 115, 20, 12]  [95, 51, 20, 12]   [98, 87, 20, 12]   
452359  [100, 115, 20, 12]  [95, 51, 20, 12]   [97, 86, 20, 12]   

                    pred     union                                 
                       0  ic_multi  ic_ratio ic@000 ic@030 ic@050  
0         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
1         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
2         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
3         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
4         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
...                  ...       ...       ...    ...    ...    ...  
452355  [98, 86, 20, 12]  0.466667  0.466667    1.0    1.0    0.0  
452356  [98, 86, 20, 12]  0.566667  0.566667    1.0    1.0    1.0  
452357  [98, 86, 20, 12]  0.675000  0.675000    1.0    1.0    1.0  
452358  [98, 86, 20, 12]  0.916667  0.916667    1.0    1.0    1.0  
452359  [98, 86, 20, 12]  0.950000  0.950000    1.0    1.0    1.0  

[452360 rows x 13 columns]

In [63]:
gt_cols = result_inference_vanilla.columns[
    out_vanilla.columns.get_level_values(0).str.startswith("union")
]

g_vanilla = out_vanilla.groupby(("meta", "replay"))[gt_cols].mean()

out_per_replay_vanilla = g_vanilla.T.groupby(level=1).mean().T

out_per_replay_vanilla.loc[['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep']]

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
275.rep,0.744752,0.710218,0.670600,1.389356,0.603162
1725.rep,0.832032,0.808940,0.782164,1.593135,0.707757
3613.rep,0.815394,0.796073,0.770766,1.532221,0.694520
4520.rep,0.783689,0.754540,0.717846,1.376201,0.641046
4664.rep,0.762503,0.737086,0.711814,1.462171,0.644365


In [68]:
out_per_replay_vanilla.loc[['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep']].describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.787674,0.761371,0.730638,1.470617,0.658170
std,0.036152,0.040988,0.045792,0.092729,0.042690
min,0.744752,0.710218,0.670600,1.376201,0.603162
25%,0.762503,0.737086,0.711814,1.389356,0.641046
50%,0.783689,0.754540,0.717846,1.462171,0.644365
75%,0.815394,0.796073,0.770766,1.532221,0.694520
max,0.832032,0.808940,0.782164,1.593135,0.707757


In [71]:
out_per_replay_vanilla.loc[~out_per_replay_vanilla.index.isin(['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep'])].describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,8.000000,8.000000,8.000000,8.000000,8.000000
mean,0.320501,0.289024,0.265382,0.327719,0.236642
std,0.082397,0.082277,0.075728,0.122775,0.068060
min,0.252755,0.193768,0.152436,0.198402,0.146029
25%,0.255616,0.230252,0.219299,0.248215,0.193516
50%,0.292743,0.271741,0.258427,0.284455,0.224463
75%,0.352122,0.323755,0.300280,0.378742,0.268490
max,0.459574,0.427494,0.387030,0.530466,0.351070


In [64]:
gt_cols = result_inference_kbrs.columns[
    out_kbrs.columns.get_level_values(0).str.startswith("union")
]

g_kbrs = out_kbrs.groupby(("meta", "replay"))[gt_cols].mean()

out_per_replay_kbrs = g_kbrs.T.groupby(level=1).mean().T

out_per_replay_kbrs.loc[['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep']]

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
275.rep,0.785539,0.757258,0.723234,1.535612,0.660040
1725.rep,0.836541,0.817242,0.779470,1.654330,0.725188
3613.rep,0.839021,0.807414,0.778536,1.627390,0.720829
4520.rep,0.839456,0.814993,0.783440,1.546006,0.708312
4664.rep,0.804204,0.789563,0.761440,1.565070,0.700332


In [67]:
out_per_replay_kbrs.loc[['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep']].describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.820952,0.797294,0.765224,1.585681,0.702940
std,0.024731,0.024881,0.024947,0.052341,0.025938
min,0.785539,0.757258,0.723234,1.535612,0.660040
25%,0.804204,0.789563,0.761440,1.546006,0.700332
50%,0.836541,0.807414,0.778536,1.565070,0.708312
75%,0.839021,0.814993,0.779470,1.627390,0.720829
max,0.839456,0.817242,0.783440,1.654330,0.725188


In [73]:
out_per_replay_kbrs.loc[~out_per_replay_kbrs.index.isin(['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep'])].describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.870834,0.846070,0.816116,1.651812,0.747736
std,0.036341,0.043678,0.049691,0.252359,0.051096
min,0.824650,0.786599,0.756265,1.237885,0.686046
25%,0.844120,0.811433,0.774894,1.457108,0.708091
50%,0.871455,0.841534,0.807795,1.635522,0.734160
75%,0.890597,0.873761,0.854287,1.900476,0.790711
max,0.925148,0.909745,0.884328,1.932637,0.816681


In [66]:
# def compute_total_intersection_percent(out: pd.DataFrame):
#     if ("meta", "replay") not in out.columns:
#         if out.index.names and "meta" in out.index.names and "replay" in out.index.names:
#             out = out.reset_index()
#         else:
#             raise KeyError("('meta','replay')가 컬럼/인덱스 어디에도 없습니다.")

#     if ("union", "ic_ratio") not in out.columns:
#         raise KeyError("('union','ic_ratio') 컬럼이 없습니다. add_union_metrics_parallel 결과를 확인하세요.")

#     overall_total = float(out[("union", "ic_ratio")].mean())

#     g = out.groupby(("meta", "replay"))[[("union", "ic_ratio")]].mean()
#     out_per_replay = g.T.groupby(level=1).mean().T
#     out_per_replay = out_per_replay.rename(columns={"ic_ratio": "total_intersection_percent"})

#     return overall_total, out_per_replay

In [ ]:
# compute_total_intersection_percent(out)

In [89]:
top5_annotation_prediction_kbrs = dataframe_topk_score(annotation_prediction_kbrs, K=5)

In [90]:
top5_annotation_prediction_kbrs["idx"] = top5_annotation_prediction_kbrs.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction_kbrs_top5 = top5_annotation_prediction_kbrs.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction_kbrs_top5.columns = [f"pred_{i}" for i in wide_prediction_kbrs_top5.columns]
wide_prediction_kbrs_top5.reset_index(inplace=True)

In [91]:
result_inference_kbrs_top5 = pd.merge(wide_ground_truth, wide_prediction_kbrs_top5, on=["replay", "image_id"], how="outer")
result_inference_kbrs_top5 = result_inference_kbrs_top5.bfill()
result_inference_kbrs_top5 = result_inference_kbrs_top5.dropna()

new_cols = []
for c in result_inference_kbrs_top5.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback
        
result_inference_kbrs_top5.columns = pd.MultiIndex.from_tuples(new_cols)
result_inference_kbrs_top5.head()

meta                         gt                                      \
      replay image_id                 0                 1                 2   
0  11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                 pred                      \
                  3                 4               0                   1   
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  [107, 112, 20, 12]   
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  [107, 112, 20, 12]   
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  [107, 112, 20, 12]   
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  [107, 112, 20, 12]   
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  [107, 112, 20, 12]   

                                                         
                2                   3                 4  
0  [2, 5, 20, 12]  [106, 108, 20, 12]  [107, 1, 20, 12]  
1  [2, 5, 20, 12]  [106, 108, 20, 12]  [107, 1, 20, 12]  
2  [2, 5, 20, 12]  [106, 108, 20, 12]  [107, 1, 20, 12]  
3  [2, 5, 20, 12]  [106, 108, 20, 12]  [107, 1, 20, 12]  
4  [2, 5, 20, 12]  [106, 108, 20, 12]  [107, 1, 20, 12]

In [92]:
out_kbrs_top5 = add_union_metrics_parallel(
    result_inference_kbrs_top5,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"), ("pred", "1"), ("pred", "2")),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

In [93]:
out_kbrs_top5

meta                          gt                    \
           replay image_id                  0                 1   
0       11251.rep        0   [108, 2, 20, 12]  [108, 2, 20, 12]   
1       11251.rep        1   [108, 2, 20, 12]  [108, 2, 20, 12]   
2       11251.rep        2   [108, 2, 20, 12]  [108, 2, 20, 12]   
3       11251.rep        3   [108, 2, 20, 12]  [108, 2, 20, 12]   
4       11251.rep        4   [108, 2, 20, 12]  [108, 2, 20, 12]   
...           ...      ...                ...               ...   
452355   6219.rep    36788  [104, 48, 20, 12]    [0, 1, 20, 12]   
452356   6219.rep    36789  [104, 48, 20, 12]    [0, 1, 20, 12]   
452357   6219.rep    36790  [104, 48, 20, 12]    [0, 1, 20, 12]   
452358   6219.rep    36791  [104, 48, 20, 12]    [0, 1, 20, 12]   
452359   6219.rep    36792  [104, 48, 20, 12]    [0, 1, 20, 12]   

                                                                 \
                         2                 3                  4   
0         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
1         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
2         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
3         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
4         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
...                    ...               ...                ...   
452355  [100, 115, 20, 12]  [95, 51, 20, 12]  [102, 91, 20, 12]   
452356  [100, 115, 20, 12]  [95, 51, 20, 12]  [101, 90, 20, 12]   
452357  [100, 115, 20, 12]  [95, 51, 20, 12]  [100, 89, 20, 12]   
452358  [100, 115, 20, 12]  [95, 51, 20, 12]   [98, 87, 20, 12]   
452359  [100, 115, 20, 12]  [95, 51, 20, 12]   [97, 86, 20, 12]   

                    pred                                         \
                       0                   1                  2   
0         [0, 2, 19, 12]  [107, 112, 20, 12]     [2, 5, 20, 12]   
1         [0, 2, 19, 12]  [107, 112, 20, 12]     [2, 5, 20, 12]   
2         [0, 2, 19, 12]  [107, 112, 20, 12]     [2, 5, 20, 12]   
3         [0, 2, 19, 12]  [107, 112, 20, 12]     [2, 5, 20, 12]   
4         [0, 2, 19, 12]  [107, 112, 20, 12]     [2, 5, 20, 12]   
...                  ...                 ...                ...   
452355  [98, 86, 20, 12]   [102, 49, 20, 12]  [108, 51, 19, 12]   
452356  [98, 86, 20, 12]   [102, 49, 20, 12]  [108, 51, 19, 12]   
452357  [98, 86, 20, 12]   [102, 49, 20, 12]  [108, 51, 19, 12]   
452358  [98, 86, 20, 12]   [105, 49, 20, 12]   [93, 88, 20, 12]   
452359  [98, 86, 20, 12]   [105, 49, 20, 12]  [107, 46, 20, 12]   

                                                  union                   \
                         3                  4  ic_multi  ic_ratio ic@000   
0       [106, 108, 20, 12]   [107, 1, 20, 12]  0.000000  0.000000    0.0   
1       [106, 108, 20, 12]   [107, 1, 20, 12]  0.000000  0.000000    0.0   
2       [106, 108, 20, 12]   [107, 1, 20, 12]  0.000000  0.000000    0.0   
3       [106, 108, 20, 12]   [107, 1, 20, 12]  0.000000  0.000000    0.0   
4       [106, 108, 20, 12]   [107, 1, 20, 12]  0.000000  0.000000    0.0   
...                    ...                ...       ...       ...    ...   
452355   [107, 46, 20, 12]   [93, 88, 20, 12]  0.813793  0.643103    1.0   
452356   [107, 46, 20, 12]   [93, 88, 20, 12]  0.855172  0.684483    1.0   
452357   [107, 46, 20, 12]   [93, 88, 20, 12]  0.900000  0.729310    1.0   
452358    [99, 51, 20, 12]  [65, 111, 20, 13]  0.954386  0.796491    1.0   
452359    [93, 88, 20, 12]   [99, 51, 20, 12]  0.992832  0.831541    1.0   

                      
       ic@030 ic@050  
0         0.0    0.0  
1         0.0    0.0  
2         0.0    0.0  
3         0.0    0.0  
4         0.0    0.0  
...       ...    ...  
452355    1.0    1.0  
452356    1.0    1.0  
452357    1.0    1.0  
452358    1.0    1.0  
452359    1.0    1.0  

[452360 rows x 17 columns]

In [94]:
gt_cols = result_inference_kbrs_top5.columns[
    out_kbrs_top5.columns.get_level_values(0).str.startswith("union")
]

g_kbrs = out_kbrs_top5.groupby(("meta", "replay"))[gt_cols].mean()

out_per_replay_kbrs_top5 = g_kbrs.T.groupby(level=1).mean().T

out_per_replay_kbrs_top5.loc[['275.rep', '1725.rep', '3613.rep', '4520.rep', '4664.rep']]

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
275.rep,0.953410,0.845474,0.550774,1.075667,0.521121
1725.rep,0.974929,0.908896,0.662910,1.131231,0.577305
3613.rep,0.984564,0.885855,0.629581,1.101608,0.562668
4520.rep,0.969566,0.886848,0.631147,1.082802,0.564047
4664.rep,0.969944,0.889490,0.598405,1.053004,0.545082


In [97]:
import numpy as np
import pandas as pd

def add_iou_columns(
    df: pd.DataFrame,
    gt_key: str = "gt",
    pred_key: str = "pred",
    thresholds=(0.0, 0.3, 0.5),
    attach: bool = True
):
    # gt/pred에 공통으로 존재하는 pair 키(예: 0..4)
    keys = [k for k in df[gt_key].columns if k in df[pred_key].columns]
    if not keys:
        raise ValueError("No overlapping ('gt', k) and ('pred', k) columns.")

    out = {}
    for k in keys:
        # 각 셀: [x, y, w, h] (top-left, width, height)
        gt   = np.asarray(df[(gt_key,   k)].to_list(), dtype=float)  # (N,4)
        pred = np.asarray(df[(pred_key, k)].to_list(), dtype=float)  # (N,4)

        xg, yg, wg, hg = gt.T
        xp, yp, wp, hp = pred.T

        # x2,y2
        xg2, yg2 = xg + wg, yg + hg
        xp2, yp2 = xp + wp, yp + hp

        # 교집합
        ix1 = np.maximum(xg,  xp)
        iy1 = np.maximum(yg,  yp)
        ix2 = np.minimum(xg2, xp2)
        iy2 = np.minimum(yg2, yp2)
        iw  = np.clip(ix2 - ix1, 0, None)
        ih  = np.clip(iy2 - iy1, 0, None)
        inter = iw * ih

        # 합집합
        area_g = np.clip(wg, 0, None) * np.clip(hg, 0, None)
        area_p = np.clip(wp, 0, None) * np.clip(hp, 0, None)
        union  = area_g + area_p - inter

        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

        # IoU 값
        out[("iou", k, "value")] = iou

        # 임계값별 일치 여부
        for thr in thresholds:
            tag = f"iou@{int(round(thr*100)):03d}"  # 0.3 -> 'iou@030'
            out[("iou", k, tag)] = iou >= thr

    iou_df = pd.DataFrame(out)
    iou_df.columns = pd.MultiIndex.from_tuples(iou_df.columns, names=["metric","pair","comp"])
    return pd.concat([df, iou_df], axis=1) if attach else iou_df

In [98]:
# 사용 예
# df = ...  # 스크린샷과 같은 멀티인덱스 컬럼 DataFrame
df_iou = add_iou_columns(out_kbrs_top5, thresholds=(0.0, 0.3, 0.5))

In [126]:
import numpy as np
import pandas as pd

def add_pairwise_iou(
    df: pd.DataFrame,
    gt_key: str = "gt",
    pred_key: str = "pred",
    ensure_gt=None,                 # 예: range(5) -> gt 0..4 강제 생성
    ensure_pred=None,               # 예: range(6) -> pred 0..5 강제 생성
    thresholds=(0.0, 0.3, 0.5),     # 임계값 bool 컬럼도 생성(0.0은 inter>0)
    fill_invalid=np.nan,            # 유효치 없을 때 채울 값 (np.nan 권장)
    out_key: str = "iou",           # IoU 값 컬럼의 레벨0 이름
):
    # --- 서브키 수집 ---
    def _subs(top):
        if top in df.columns.get_level_values(0):
            return list(df[top].columns)
        return []

    def _kmap(subs):  # "표기 문자열" -> "원래 키"
        return {str(k): k for k in subs}

    gt_subs, pred_subs = _subs(gt_key), _subs(pred_key)
    gmap, pmap = _kmap(gt_subs), _kmap(pred_subs)

    def _sort_keys_str(keys):
        return sorted(keys, key=lambda s: (0, int(s)) if s.isdigit() else (1, s))

    if ensure_gt is not None:
        gt_names_str = [str(k) for k in ensure_gt]
    else:
        gt_names_str = _sort_keys_str(list(gmap.keys()))

    if ensure_pred is not None:
        pred_names_str = [str(k) for k in ensure_pred]
    else:
        pred_names_str = _sort_keys_str(list(pmap.keys()))

    N, G, P = len(df), len(gt_names_str), len(pred_names_str)

    # --- Series -> (N,4) 변환 ---
    def _to_xywh(series):
        if series is None:
            return np.full((N, 4), np.nan, dtype=float)
        try:
            arr = np.asarray(series.to_list(), dtype=float)
            if arr.ndim != 2 or arr.shape[1] < 4:
                raise ValueError
            return arr[:, :4]
        except Exception:
            arr = np.full((N, 4), np.nan, dtype=float)
            vals = series.to_list()
            for i, v in enumerate(vals):
                if isinstance(v, (list, tuple, np.ndarray)) and len(v) >= 4:
                    try:
                        arr[i, :4] = np.array(v[:4], dtype=float)
                    except Exception:
                        pass
            return arr

    # --- gt, pred 배열 구성: (N,G,4), (N,P,4) ---
    g = np.full((N, G, 4), np.nan, dtype=float)
    p = np.full((N, P, 4), np.nan, dtype=float)

    for gi, gs in enumerate(gt_names_str):
        s = df[(gt_key, gmap[gs])] if gs in gmap else None
        g[:, gi, :] = _to_xywh(s)

    for pj, ps in enumerate(pred_names_str):
        s = df[(pred_key, pmap[ps])] if ps in pmap else None
        p[:, pj, :] = _to_xywh(s)

    # 유효 여부 (각 박스 4요소 모두 수치)
    g_valid = ~np.isnan(g).any(axis=2)   # (N,G)
    p_valid = ~np.isnan(p).any(axis=2)   # (N,P)
    valid_pair = g_valid[:, :, None] & p_valid[:, None, :]  # (N,G,P)

    # --- 브로드캐스팅으로 (N,G,P) IoU 계산 ---
    wg = np.clip(g[:, :, 2], 0, None)[:, :, None]
    hg = np.clip(g[:, :, 3], 0, None)[:, :, None]
    wp = np.clip(p[:, :, 2], 0, None)[:, None, :]
    hp = np.clip(p[:, :, 3], 0, None)[:, None, :]

    xg1 = g[:, :, 0][:, :, None]; yg1 = g[:, :, 1][:, :, None]
    xp1 = p[:, :, 0][:, None, :]; yp1 = p[:, :, 1][:, None, :]
    xg2 = xg1 + wg; yg2 = yg1 + hg
    xp2 = xp1 + wp; yp2 = yp1 + hp

    ix1 = np.maximum(xg1, xp1)
    iy1 = np.maximum(yg1, yp1)
    ix2 = np.minimum(xg2, xp2)
    iy2 = np.minimum(yg2, yp2)

    iw = np.clip(ix2 - ix1, 0, None)
    ih = np.clip(iy2 - iy1, 0, None)
    inter = iw * ih
    area_g = wg * hg
    area_p = wp * hp
    union = area_g + area_p - inter

    iou = np.full((N, G, P), fill_invalid, dtype=float)
    can = valid_pair & (union > 0)
    iou[can] = inter[can] / union[can]

    # --- 컬럼 생성 (2-레벨: metric, pair) ---
    out = {}
    for gi, gs in enumerate(gt_names_str):
        gt_lbl = int(gs) if gs.isdigit() else gs
        for pj, ps in enumerate(pred_names_str):
            pr_lbl = int(ps) if ps.isdigit() else ps
            pair_label = f"g{gt_lbl}_p{pr_lbl}"

            # IoU 실수값
            out[(out_key, pair_label)] = iou[:, gi, pj]

            # # 임계값 bool (0.0은 "조금이라도 겹치면": inter>0)
            # if thresholds is not None:
            #     for thr in thresholds:
            #         name = f"iou@{int(round(thr*100)):03d}"
            #         mask = np.zeros(N, dtype=bool)
            #         if thr == 0.0:
            #             cond = valid_pair[:, gi, pj] & (inter[:, gi, pj] > 0)
            #         else:
            #             cond = valid_pair[:, gi, pj] & (iou[:, gi, pj] >= thr)
            #         mask[cond] = True
            #         out[(name, pair_label)] = mask

    out_df = pd.DataFrame(out)
    out_df.columns = pd.MultiIndex.from_tuples(out_df.columns, names=["metric", "pair"])
    return pd.concat([df, out_df], axis=1)


In [127]:
# 사용 예
# df = ...  # ('gt',0..4), ('pred',0..4) 형태의 멀티인덱스 컬럼 DataFrame
df2 = add_pairwise_iou(out_kbrs_top5)             # ('iou', k) 컬럼들이 추가됨
# pair0_iou = df2[("iou", 0)].head()   # pair=0의 IoU 값 확인

In [145]:
df2[['meta', 'iou']]

meta            iou                                          \
           replay image_id g0_p0     g0_p1     g0_p2     g0_p3     g0_p4   
0       11251.rep        0   0.0  0.000000  0.000000  0.000000  0.771218   
1       11251.rep        1   0.0  0.000000  0.000000  0.000000  0.771218   
2       11251.rep        2   0.0  0.000000  0.000000  0.000000  0.771218   
3       11251.rep        3   0.0  0.000000  0.000000  0.000000  0.771218   
4       11251.rep        4   0.0  0.000000  0.000000  0.000000  0.771218   
...           ...      ...   ...       ...       ...       ...       ...   
452355   6219.rep    36788   0.0  0.702128  0.444444  0.548387  0.000000   
452356   6219.rep    36789   0.0  0.702128  0.444444  0.548387  0.000000   
452357   6219.rep    36790   0.0  0.702128  0.444444  0.548387  0.000000   
452358   6219.rep    36791   0.0  0.771218  0.000000  0.391304  0.000000   
452359   6219.rep    36792   0.0  0.771218  0.548387  0.000000  0.391304   

                          ...                                                \
       g1_p0 g1_p1 g1_p2  ... g3_p0     g3_p1     g3_p2     g3_p3     g3_p4   
0        0.0   0.0   0.0  ...   0.0  0.000000  0.000000  0.000000  0.771218   
1        0.0   0.0   0.0  ...   0.0  0.000000  0.000000  0.000000  0.771218   
2        0.0   0.0   0.0  ...   0.0  0.000000  0.000000  0.000000  0.771218   
3        0.0   0.0   0.0  ...   0.0  0.000000  0.000000  0.000000  0.771218   
4        0.0   0.0   0.0  ...   0.0  0.000000  0.000000  0.000000  0.771218   
...      ...   ...   ...  ...   ...       ...       ...       ...       ...   
452355   0.0   0.0   0.0  ...   0.0  0.371429  0.218750  0.132075  0.000000   
452356   0.0   0.0   0.0  ...   0.0  0.371429  0.218750  0.132075  0.000000   
452357   0.0   0.0   0.0  ...   0.0  0.371429  0.218750  0.132075  0.000000   
452358   0.0   0.0   0.0  ...   0.0  0.263158  0.000000  0.666667  0.000000   
452359   0.0   0.0   0.0  ...   0.0  0.263158  0.132075  0.000000  0.666667   

                                                 
           g4_p0 g4_p1    g4_p2 g4_p3     g4_p4  
0       0.000000   0.0  0.00000   0.0  0.771218  
1       0.000000   0.0  0.00000   0.0  0.771218  
2       0.000000   0.0  0.00000   0.0  0.771218  
3       0.000000   0.0  0.00000   0.0  0.771218  
4       0.000000   0.0  0.00000   0.0  0.771218  
...          ...   ...      ...   ...       ...  
452355  0.304348   0.0  0.00000   0.0  0.259843  
452356  0.395349   0.0  0.00000   0.0  0.333333  
452357  0.509434   0.0  0.00000   0.0  0.424332  
452358  0.846154   0.0  0.52381   0.0  0.000000  
452359  0.904762   0.0  0.00000   0.5  0.000000  

[452360 rows x 27 columns]

In [137]:
df2['iou'].loc[~df2['iou'].fillna(0).eq(0).all(axis=1)]

,g0_p0,g0_p1,g0_p2,g0_p3,g0_p4,g1_p0,g1_p1,g1_p2,g1_p3,g1_p4,...,g3_p0,g3_p1,g3_p2,g3_p3,g3_p4,g4_p0,g4_p1,g4_p2,g4_p3,g4_p4
0,0.0,0.000000,0.000000,0.000000,0.771218,0.0,0.0,0.0,0.0,0.771218,...,0.0,0.000000,0.000000,0.000000,0.771218,0.000000,0.0,0.00000,0.0,0.771218
1,0.0,0.000000,0.000000,0.000000,0.771218,0.0,0.0,0.0,0.0,0.771218,...,0.0,0.000000,0.000000,0.000000,0.771218,0.000000,0.0,0.00000,0.0,0.771218
2,0.0,0.000000,0.000000,0.000000,0.771218,0.0,0.0,0.0,0.0,0.771218,...,0.0,0.000000,0.000000,0.000000,0.771218,0.000000,0.0,0.00000,0.0,0.771218
3,0.0,0.000000,0.000000,0.000000,0.771218,0.0,0.0,0.0,0.0,0.771218,...,0.0,0.000000,0.000000,0.000000,0.771218,0.000000,0.0,0.00000,0.0,0.771218
4,0.0,0.000000,0.000000,0.000000,0.771218,0.0,0.0,0.0,0.0,0.771218,...,0.0,0.000000,0.000000,0.000000,0.771218,0.000000,0.0,0.00000,0.0,0.771218
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452355,0.0,0.702128,0.444444,0.548387,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.371429,0.218750,0.132075,0.000000,0.304348,0.0,0.00000,0.0,0.259843
452356,0.0,0.702128,0.444444,0.548387,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.371429,0.218750,0.132075,0.000000,0.395349,0.0,0.00000,0.0,0.333333
452357,0.0,0.702128,0.444444,0.548387,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.371429,0.218750,0.132075,0.000000,0.509434,0.0,0.00000,0.0,0.424332
452358,0.0,0.771218,0.000000,0.391304,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.263158,0.000000,0.666667,0.000000,0.846154,0.0,0.52381,0.0,0.000000


In [140]:
len(df2['iou'].loc[~df2['iou'].fillna(0).eq(0).all(axis=1)]) / len(df2['iou'])

0.9934985409850562

In [146]:
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) 'iou' 멀티컬럼 -> (N,G,P) 큐브로 변환
#    df.columns level0='iou', level1='g{i}_p{j}' 형식 가정
# ------------------------------------------------------------
def _extract_iou_cube(df, iou_key="iou"):
    iou_df = df.xs(iou_key, axis=1, level=0)  # pair 라벨 컬럼만
    pair_cols = list(iou_df.columns)

    def _parse_pair(c):
        m = re.match(r"g(\d+)_p(\d+)", str(c))
        return (int(m.group(1)), int(m.group(2))) if m else None

    parsed = [_parse_pair(c) for c in pair_cols]
    g_ids = sorted({g for gp in parsed if gp for g in [gp[0]]})
    p_ids = sorted({p for gp in parsed if gp for p in [gp[1]]})
    g_index = {g:i for i,g in enumerate(g_ids)}
    p_index = {p:j for j,p in enumerate(p_ids)}

    N, G, P = len(iou_df), len(g_ids), len(p_ids)
    cube = np.full((N, G, P), np.nan, dtype=float)

    for col, gp in zip(pair_cols, parsed):
        if gp is None: 
            continue
        gi, pj = g_index[gp[0]], p_index[gp[1]]
        cube[:, gi, pj] = iou_df[col].to_numpy(dtype=float)

    return cube, g_ids, p_ids

# ------------------------------------------------------------
# (A) Pair별 요약 테이블: mean IoU, hit rate@t, count
# ------------------------------------------------------------
def summarize_pairs(df, thresholds=(0.0, 0.3, 0.5)):
    cube, g_ids, p_ids = _extract_iou_cube(df)
    G, P = len(g_ids), len(p_ids)

    mean_iou = np.nanmean(cube, axis=0)  # (G,P)
    count    = np.sum(~np.isnan(cube), axis=0)

    # t별 hit rate (t=0.0은 IoU>0)
    hit_rates = {}
    for t in thresholds:
        if t == 0.0:
            hits = (cube > 0)
        else:
            hits = (cube >= t)
        hit = np.nanmean(hits.astype(float), axis=0)  # 평균 = 비율
        hit_rates[t] = hit

    idx = [f"g{g}" for g in g_ids]
    cols = [f"p{p}" for p in p_ids]
    out = {
        "mean_iou": pd.DataFrame(mean_iou, index=idx, columns=cols),
        "count":    pd.DataFrame(count,    index=idx, columns=cols),
    }
    for t, mat in hit_rates.items():
        out[f"hit@{int(round(t*100)):03d}"] = pd.DataFrame(mat, index=idx, columns=cols)
    return out

# ------------------------------------------------------------
# (B) Best-of 매칭(중복 허용): GT Recall@t, Pred Precision@t, F1@t
#   - GT recall@t: 각 (n,g)에서 max_j IoU >= t
#   - Pred precision@t: 각 (n,p)에서 max_i IoU >= t
#   - 분모는 해당 행에서 "유효한 gt/pred"만 포함 (전부 NaN 제외)
# ------------------------------------------------------------
def bestof_metrics(df, thresholds=(0.0, 0.3, 0.5)):
    cube, g_ids, p_ids = _extract_iou_cube(df)
    N, G, P = cube.shape

    # 유효 gt/pred (해당 축에 NaN만 있으면 비유효)
    gt_present   = ~np.isnan(cube).all(axis=2)  # (N,G)
    pred_present = ~np.isnan(cube).all(axis=1)  # (N,P)

    res = {}
    for t in thresholds:
        if t == 0.0:
            gt_hit   = (np.nanmax(cube, axis=2) > 0)   # (N,G)
            pred_hit = (np.nanmax(cube, axis=1) > 0)   # (N,P)
        else:
            gt_hit   = (np.nanmax(cube, axis=2) >= t)
            pred_hit = (np.nanmax(cube, axis=1) >= t)

        # 마스크로 분모/분자 집계 (NaN row 제외)
        gt_recall = (gt_hit & gt_present).sum() / np.maximum(gt_present.sum(), 1)
        pr_prec   = (pred_hit & pred_present).sum() / np.maximum(pred_present.sum(), 1)
        f1 = 0.0
        if gt_recall + pr_prec > 0:
            f1 = 2 * gt_recall * pr_prec / (gt_recall + pr_prec)

        res[t] = dict(recall_gt=gt_recall, precision_pred=pr_prec, f1=f1)
    return res

# ------------------------------------------------------------
# (C) 일대일 매칭(Hungarian/그리디)
#   - 각 행 n에서 GT×Pred IoU 행렬에서 t 이상만 후보로 1:1 매칭
#   - 행마다 매칭된 페어의 개수/IoU를 합산하여 전체 Recall/Precision/F1, mIoU
#   - SciPy가 없으면 greedy(내림차순)로 대체
# ------------------------------------------------------------
try:
    from scipy.optimize import linear_sum_assignment
except Exception:
    linear_sum_assignment = None

def one2one_metrics(df, thresholds=(0.0, 0.3, 0.5)):
    cube, g_ids, p_ids = _extract_iou_cube(df)
    N, G, P = cube.shape
    metrics = {}

    for t in thresholds:
        matched = 0
        iou_sum = 0.0
        denom_gt = 0  # 전체 gt 유효 개수의 합
        denom_pr = 0  # 전체 pred 유효 개수의 합

        for n in range(N):
            M = cube[n]  # (G,P)
            # 유효한 gt/pred 만 고려
            g_mask = ~np.isnan(M).all(axis=1)
            p_mask = ~np.isnan(M).all(axis=0)
            Gi = np.where(g_mask)[0]
            Pj = np.where(p_mask)[0]

            denom_gt += len(Gi)
            denom_pr += len(Pj)
            if len(Gi) == 0 or len(Pj) == 0:
                continue

            subM = M[np.ix_(Gi, Pj)]  # (g', p')

            # 임계 미만은 후보 제외
            if t == 0.0:
                cand = (subM > 0)
            else:
                cand = (subM >= t)

            if not np.any(cand):
                continue

            if linear_sum_assignment is not None:
                # invalid에 큰 cost, 나머지는 cost=1-IoU
                cost = np.where(cand, 1.0 - np.clip(subM, 0, 1), 1e6)
                ri, cj = linear_sum_assignment(cost)
                for r, c in zip(ri, cj):
                    v = subM[r, c]
                    if (t == 0.0 and v > 0) or (t > 0.0 and v >= t):
                        matched += 1
                        iou_sum += float(v)
            else:
                # Greedy: IoU 내림차순으로 GT/Pred 중복 배제
                items = [(i, j, subM[i, j]) for i in range(subM.shape[0]) for j in range(subM.shape[1])]
                items = [x for x in items if (t == 0.0 and x[2] > 0) or (t > 0.0 and x[2] >= t)]
                items.sort(key=lambda x: x[2], reverse=True)
                used_r, used_c = set(), set()
                for r, c, v in items:
                    if r in used_r or c in used_c:
                        continue
                    used_r.add(r); used_c.add(c)
                    matched += 1
                    iou_sum += float(v)

        recall = matched / max(denom_gt, 1)
        prec   = matched / max(denom_pr, 1)
        f1 = 0.0
        if recall + prec > 0:
            f1 = 2 * recall * prec / (recall + prec)
        miou = (iou_sum / matched) if matched > 0 else 0.0
        metrics[t] = dict(recall_gt=recall, precision_pred=prec, f1=f1, mIoU=miou, matched=matched)

    return metrics


In [149]:
pair_summ = summarize_pairs(df2, thresholds=(0.0, 0.3, 0.5))
pair_summ["mean_iou"]        # g×p 평균 IoU



,p0,p1,p2,p3,p4
g0,0.271998,0.198064,0.144486,0.114831,0.101010
g1,0.244361,0.177411,0.128223,0.107520,0.095296
g2,0.293195,0.199253,0.143425,0.114980,0.099330
g3,0.259725,0.185505,0.140612,0.112106,0.097126
g4,0.262453,0.187659,0.140645,0.113020,0.099643


In [157]:
pair_summ["hit@000"]         # g×p hit rate@0.5

,p0,p1,p2,p3,p4
g0,0.498802,0.420064,0.341001,0.286327,0.258186
g1,0.425243,0.362658,0.296310,0.255717,0.231981
g2,0.492544,0.399377,0.323426,0.274255,0.244089
g3,0.468536,0.390371,0.327250,0.275298,0.245143
g4,0.477184,0.397887,0.329702,0.280500,0.253132


In [151]:
pair_summ["count"]           # 비교에 사용된 샘플 수


,p0,p1,p2,p3,p4
g0,452360,452360,452360,452360,452360
g1,452360,452360,452360,452360,452360
g2,452360,452360,452360,452360,452360
g3,452360,452360,452360,452360,452360
g4,452360,452360,452360,452360,452360


In [152]:
bo = bestof_metrics(df2, thresholds=(0.0, 0.3, 0.5))


In [153]:
bo

{0.0: {'recall_gt': np.float64(0.7919829339464144),
  'precision_pred': np.float64(0.688537005924485),
  'f1': np.float64(0.7366460165748311)},
 0.3: {'recall_gt': np.float64(0.7125395702537802),
  'precision_pred': np.float64(0.5990242284905827),
  'f1': np.float64(0.6508695448119413)},
 0.5: {'recall_gt': np.float64(0.606028384472544),
  'precision_pred': np.float64(0.4699610929348307),
  'f1': np.float64(0.5293913516747455)}}

In [154]:
m11 = one2one_metrics(df2, thresholds=(0.0, 0.3, 0.5))


In [155]:
m11

{0.0: {'recall_gt': 0.5983813776638076,
  'precision_pred': 0.5983813776638076,
  'f1': 0.5983813776638076,
  'mIoU': 0.6082953278318641,
  'matched': 1353419},
 0.3: {'recall_gt': 0.5328119197099656,
  'precision_pred': 0.5328119197099656,
  'f1': 0.5328119197099656,
  'mIoU': 0.6669346735218301,
  'matched': 1205114},
 0.5: {'recall_gt': 0.4395534530020338,
  'precision_pred': 0.4395534530020338,
  'f1': 0.4395534530020338,
  'mIoU': 0.7316876377165001,
  'matched': 994182}}